<a href="https://colab.research.google.com/github/SheffieldML/GPc/blob/main/python/examples/toy_1d_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Toy 1-D GP regression with `gpc`

CIP-001D Phase E + CIP-0031 predict contract. Build the extension first (`cd python && make`), then run this notebook with that environment.

Prefer **noisy** observations: noiseless \(Y\) drives `likelihood.variance` → 0 and near-zero **latent** variance at the training points.

Prediction (GPy names):
- `predict(X)` → mean/var of \(y_*\) (includes observation noise)
- `predict_noiseless(X)` → mean/var of \(f_*\)

For Gaussian noise, `var_y - var_f` equals `likelihood.variance`.

In [ ]:
%pip install --no-cache-dir --upgrade --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "gpc==0.1.0a9"


In [ ]:
%pip install matplotlib ipywidgets


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import sys
from pathlib import Path

# Editable layout: python/ on sys.path when running from repo examples/
root = Path.cwd()
if (root / "gpc").is_dir():
    sys.path.insert(0, str(root))
elif (root.parent / "gpc").is_dir():
    sys.path.insert(0, str(root.parent))

import numpy as np
import gpc

rng = np.random.default_rng(0)
X = np.linspace(0, 1, 25)[:, None]
Y = np.sin(2 * np.pi * X) + 0.05 * rng.normal(size=X.shape)

m = gpc.GPRegression(X, Y)
m.set_by_path("rbf.lengthscale", 0.2)
print("paths:", m.param_paths())
print("objective before:", m.objective())
# messages=True: paramz-style in-place progress (ipywidgets in Colab/Jupyter).
m.optimize(iters=50, messages=True)
print("objective after:", m.objective())
print(m)

In [ ]:
try:
    from gpc.plot import plot_1d
    import matplotlib.pyplot as plt
except ImportError:
    raise SystemExit("Install plotting extras: pip install matplotlib  (or gpc[plot])")

# Default band is y_* (observation noise included).
plot_1d(m, X, Y, label="predict (y_*)")
plt.title("predict — noisy observations")
plt.show()

## `predict` vs `predict_noiseless` (CIP-0031)

Same means; predictive variance is latent variance plus `likelihood.variance`.

In [ ]:
Xtest = np.linspace(0.0, 1.0, 200)[:, None]
mu_y, var_y = m.predict(Xtest)
mu_f, var_f = m.predict_noiseless(Xtest)
sigma2 = m.get_by_path("likelihood.variance")

print("likelihood.variance:", sigma2)
print("max |mu_y - mu_f|:", float(np.max(np.abs(mu_y - mu_f))))
print("mean(var_y - var_f):", float(np.mean(var_y - var_f)))
assert np.allclose(mu_y, mu_f)
assert np.allclose(var_y - var_f, sigma2, atol=1e-10)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
plot_1d(m, X, Y, Xtest=Xtest, ax=axes[0], label="y_*", noiseless=False)
axes[0].set_title("predict (y_*)")
plot_1d(m, X, Y, Xtest=Xtest, ax=axes[1], label="f_*", noiseless=True)
axes[1].set_title("predict_noiseless (f_*)")
fig.suptitle("Toy 1-D — observation noise widens the y_* band")
plt.tight_layout()
plt.show()